In [9]:
import os
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path

from tqdm import tqdm
import matplotlib.pyplot as plt

from rod_tracking_env import RodTrackingEnv
from training_student import StudentPolicy

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

plt.style.use("default")
plt.rcParams.update({
    "figure.facecolor":  "white",
    "axes.facecolor":    "white",
    "savefig.facecolor": "white",
})

In [10]:
# CONFIGURATION

# STUDENT_PATH = "results_student_v3/student_policy.pth"
# OUTPUT_DIR   = Path("results_student_v3/validation")

# STUDENT_PATH = "results_student_v4_noise/student_policy.pth"
# OUTPUT_DIR   = Path("results_student_v4_noise/validation")

STUDENT_PATH = "results_dagger_targeted/round_1/student_targeted_r1.pth"
OUTPUT_DIR   = Path("results_dagger_targeted/round_1/validation")


# E values to validate on: 4 trained experts + 3 interpolated points.
E_VALUES = {
    "soft":         5e6,      # esperto
    "soft_to_med":  6.5e6,    # interpolazione 5↔8
    "medium_soft":  7.5e6,    # esperto
    "med_to_med":   9e6,      # interpolazione 8↔10
    "medium":       1e7,      # esperto
    "med_to_rigid": 1.5e7,    # interpolazione 10↔20
    "rigid":        2e7,      # esperto
}

# E normalization range — MUST match generate_dataset and
# training_student.
E_MIN = 5e6
E_MAX = 2e7

# Parametri env
ENV_PARAMS = {
    'n_elem': 20,
    'sim_dt': 2.0e-4,
    'num_steps_per_update': 7,
    'base_length': 1.0,
    'base_radius': 0.05,
    'density': 1000.0,
    'NU': 11.0,
    'n_control_points': 6,
    'alpha': 75.0,
    'max_rate_of_change_of_activation': np.inf,
    'target_v_max': 0.50,
    'boundary': (-0.35, 0.35, 0.90, 1.0, -0.35, 0.35),
    'final_time': 10.0,
    'success_threshold': 0.01,
    'w_dist': 2.0,
    'w_precision': 5.0,
    'w_progress': 1.0,
    'w_smoothness': 0.03,
    'sigma_mult': 1.5,
    'sigma_floor': 0.01,
}

N_EPISODES        = 50    # episodes per (E, condition) for matched / dynamic
N_STATIC_EPISODES = 20    # static is cheaper to evaluate; fewer episodes suffice


In [11]:
def normalize_E(E: float) -> float:
    """Log-scale normalize E to [0, 1] - must match training."""
    log_E = np.log10(E)
    log_min = np.log10(E_MIN)
    log_max = np.log10(E_MAX)
    return (log_E - log_min) / (log_max - log_min)


def load_student(path: str, device: torch.device):
    """Load a trained student checkpoint.

    `weights_only=False` is explicit because we're loading a TRUSTED
    local checkpoint; newer torch versions warn about the implicit
    case for security reasons.
    """
    ckpt = torch.load(path, map_location=device)
    obs_dim    = ckpt["obs_dim"]
    action_dim = ckpt["action_dim"]
    model = StudentPolicy(obs_dim, action_dim).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    print(f"Student caricato: ep {ckpt['epoch']}, val_loss={ckpt['val_loss']:.5f}")
    print(f"  obs_dim={obs_dim}, action_dim={action_dim}")
    return model, obs_dim, action_dim


def student_predict(model, obs: np.ndarray, E_norm: float, device) -> np.ndarray:
    """Concatenate obs with E_norm, run forward, return post-tanh action.

    The student outputs PRE-tanh logits (per v3 design), so we apply
    tanh here to map to the env's [-1, 1] action range. Forgetting
    this tanh was a long-standing bug in earlier validation scripts
    - it produced spurious 'drift' between v3 and DAgger pipelines
    because the action magnitudes were inconsistent.
    """
    obs_aug = np.concatenate([obs, [E_norm]]).astype(np.float32)
    with torch.no_grad():
        x = torch.from_numpy(obs_aug).unsqueeze(0).to(device)
        action = torch.tanh(model(x)).squeeze(0).cpu().numpy()
    return action


In [12]:
# VALIDATION

def validate_student_on_E(model, E: float, n_episodes: int, device, env_params_base=None):
    """Run n_episodes at stiffness E and return aggregate metrics.

    Returns a dict with:
      - aggregate error metrics in cm (mean, median, p90, p95, std)
      - success rates at 4 thresholds (5mm, 1cm, 1.5cm, 2cm), as
        FRACTIONS in [0, 1]
      - per-episode on_goal_fraction and mean error
      - trajectories (tip + target paths) for plotting; excluded from
        the JSON dump
    """
    if env_params_base is None:
        env_params_base = ENV_PARAMS

    E_norm = normalize_E(E)
    env_params = {**env_params_base, 'young_modulus': E}
    env = RodTrackingEnv(**env_params)

    all_errors  = []   # per-step error (one entry per RL step across all eps)
    ep_on_goal  = []   # per-episode on_goal_fraction
    ep_mean_err = []   # per-episode mean error

    all_trajectories = []

    for ep in tqdm(range(n_episodes), desc=f"E={E:.1e}"):
        # Reproducible per-episode seed.
        obs, _ = env.reset(seed=ep * 7919)
        done = False
        ep_errors = []
        tip_traj, tgt_traj = [], []

        while not done:
            action = student_predict(model, obs, E_norm, device)
            obs, _, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            ep_errors.append(info["error"])
            tip_traj.append(info["tip_pos"])
            tgt_traj.append(info["target_pos"])

        all_errors.extend(ep_errors)
        ep_on_goal.append(info["on_goal_fraction"])
        ep_mean_err.append(np.mean(ep_errors))
        all_trajectories.append((
            np.array(tip_traj), np.array(tgt_traj)
        ))

    env.close()

    all_errors = np.array(all_errors)
    ep_on_goal = np.array(ep_on_goal)

    return {
        "E": E,
        "E_norm": E_norm,
        "n_episodes": n_episodes,
        "mean_error_cm": float(np.mean(all_errors) * 100),
        "median_error_cm": float(np.median(all_errors) * 100),
        "p90_error_cm": float(np.percentile(all_errors, 90) * 100),
        "p95_error_cm": float(np.percentile(all_errors, 95) * 100),
        "std_error_cm": float(np.std(all_errors) * 100),
        "success_1cm": float(np.mean(all_errors < 0.01)),
        "success_15mm": float(np.mean(all_errors < 0.015)),
        "success_2cm": float(np.mean(all_errors < 0.02)),
        "success_5mm": float(np.mean(all_errors < 0.005)),
        "on_goal_frac_mean": float(np.mean(ep_on_goal)),
        "on_goal_frac_std": float(np.std(ep_on_goal)),
        "per_ep_mean_err_cm": [float(e * 100) for e in ep_mean_err],
        "per_ep_on_goal": [float(v) for v in ep_on_goal],
        "trajectories": all_trajectories,
    }


def print_result(label: str, res: dict):
    """Print one validation result block."""
    print(f"\n{'=' * 60}")
    print(f"  {label}  (E = {res['E']:.1e})")
    print(f"{'=' * 60}")
    print(f"  Episodes:       {res['n_episodes']}")
    print(f"  Mean Error:     {res['mean_error_cm']:.2f} +/- {res['std_error_cm']:.2f} cm")
    print(f"  Median Error:   {res['median_error_cm']:.2f} cm")
    print(f"  P90 Error:      {res['p90_error_cm']:.2f} cm")
    print(f"  P95 Error:      {res['p95_error_cm']:.2f} cm")
    print(f"  Success @5mm:   {res['success_5mm']  * 100:5.1f}%")
    print(f"  Success @1cm:   {res['success_1cm']  * 100:5.1f}%")
    print(f"  Success @15mm:  {res['success_15mm'] * 100:5.1f}%")
    print(f"  Success @2cm:   {res['success_2cm']  * 100:5.1f}%")
    print(f"  OnGoal:         {res['on_goal_frac_mean']:.3f} +/- {res['on_goal_frac_std']:.3f}")


In [13]:
TRAINED_E = [(5e6, "5e6"), (7.5e6, "7.5e6"), (1e7, "1e7"), (2e7, "2e7")]

def plot_summary(results: dict, out_dir: Path,
                 suffix: str = "dyn",
                 title_prefix: str = "Student"):
    """Two-panel summary: success rates and errors vs E.

    Parameters
    ----------
    results       : dict {label -> result dict} from validate_student_on_E
    out_dir       : where to save the PNG
    suffix        : filename suffix (e.g. 'dyn' -> student_performance_dyn.png)
    title_prefix  : label inserted into the plot titles
    """
    Es  = np.array([r["E"]               for r in results.values()])
    s1  = np.array([r["success_1cm"]     for r in results.values()]) * 100
    s2  = np.array([r["success_2cm"]     for r in results.values()]) * 100
    err = np.array([r["mean_error_cm"]   for r in results.values()])
    med = np.array([r["median_error_cm"] for r in results.values()])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: success rates
    ax = axes[0]
    ax.plot(Es, s1, "o-", label="Success @1cm",
            color="#2196F3", lw=2, markersize=8)
    ax.plot(Es, s2, "s-", label="Success @2cm",
            color="#4CAF50", lw=2, markersize=8)
    ax.axhline(70, color="red", linestyle="--", alpha=0.5, label="Target 70%")
    ax.set_xscale("log")
    ax.set_xlabel("Young's modulus E [Pa]")
    ax.set_ylabel("Success rate [%]")
    ax.set_title(f"{title_prefix} performance vs E ({suffix})")
    ax.legend(loc="lower right")
    ax.grid(True, which="major", alpha=0.4)
    ax.grid(False, which="minor")
    ax.set_ylim([0, 100])

    # Vertical markers for the 4 trained experts.
    for E_train, lbl in TRAINED_E:
        ax.axvline(E_train, color="gray", linestyle=":", alpha=0.5, lw=1)
        ax.text(E_train, 95, lbl, rotation=90, fontsize=8, color="gray",
                ha="right", va="top")

    # Right: errors
    ax = axes[1]
    ax.plot(Es, err, "o-", label="Mean error",
            color="#E91E63", lw=2, markersize=8)
    ax.plot(Es, med, "s-", label="Median error",
            color="#9C27B0", lw=2, markersize=8)
    ax.axhline(1.0, color="green",  linestyle="--", alpha=0.5, label="1cm threshold")
    ax.axhline(2.0, color="orange", linestyle="--", alpha=0.5, label="2cm threshold")
    ax.set_xscale("log")
    ax.set_xlabel("Young's modulus E [Pa]")
    ax.set_ylabel("Error [cm]")
    ax.set_title(f"Mean and median error vs E ({suffix})")
    ax.legend(loc="upper right")
    ax.grid(True, which="major", alpha=0.4)
    ax.grid(False, which="minor")

    for E_train, _ in TRAINED_E:
        ax.axvline(E_train, color="gray", linestyle=":", alpha=0.5, lw=1)

    plt.tight_layout()
    out_file = out_dir / f"student_performance_{suffix}.png"
    plt.savefig(out_file, dpi=150)
    plt.show()
    print(f"  -> saved {out_file}")


def plot_error_trajectories(results: dict, out_dir: Path,
                            suffix: str = "dyn"):
    """Per-E error-over-time grid: mean +/- std across episodes."""
    fig, axes = plt.subplots(1, len(results),
                             figsize=(4 * len(results), 4),
                             sharey=True)
    if len(results) == 1:
        axes = [axes]

    for ax, (label, res) in zip(axes, results.items()):
        trajs = res["trajectories"]
        # Stack episodes into a (n_eps, max_len) matrix, padding with NaN.
        max_len = max(len(t[0]) for t in trajs)
        errors  = np.full((len(trajs), max_len), np.nan)
        for i, (tip, tgt) in enumerate(trajs):
            e = np.linalg.norm(tip - tgt, axis=1) * 100   # cm
            errors[i, :len(e)] = e

        mean  = np.nanmean(errors, axis=0)
        std   = np.nanstd(errors,  axis=0)
        steps = np.arange(max_len)

        ax.plot(steps, mean, color="#2196F3", lw=1.5, label="Mean")
        ax.fill_between(steps, mean - std, mean + std,
                        alpha=0.3, color="#2196F3")
        ax.axhline(1.0, color="green",  linestyle="--", alpha=0.5, label="1 cm")
        ax.axhline(2.0, color="orange", linestyle="--", alpha=0.5, label="2 cm")
        ax.set_title(f"{label}\nE={res['E']:.1e}")
        ax.set_xlabel("Step")
        ax.grid(True, alpha=0.3)
        if ax is axes[0]:
            ax.set_ylabel("Error [cm]")
        ax.set_ylim([0, 10])

    axes[0].legend(loc="upper right")
    plt.tight_layout()
    out_file = out_dir / f"student_error_trajectories_{suffix}.png"
    plt.savefig(out_file, dpi=150)
    plt.show()
    print(f"  -> saved {out_file}")


In [14]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}\n")

model, obs_dim, action_dim = load_student(STUDENT_PATH, device)

# ──────────────────────────────────────────────
# MATCHED VALIDATION (p_static=0.30, 50 episodes)
# ──────────────────────────────────────────────
print("\n" + "="*70)
print("  MATCHED VALIDATION (p_static=0.30)")
print("="*70)

results_match = {}
env_params_match = {**ENV_PARAMS, 'p_static': 0.30}

for label, E in E_VALUES.items():
    res = validate_student_on_E(model, E, N_EPISODES, device, env_params_match)
    print_result(label + " [MATCHED]", res)
    results_match[label] = res

# ──────────────────────────────────────────────
# DYNAMIC VALIDATION (p_static=0.0, 50 episodes)
# ──────────────────────────────────────────────
print("\n" + "="*70)
print("  DYNAMIC VALIDATION (p_static=0.0)")
print("="*70)

results_dyn = {}
env_params_dyn = {**ENV_PARAMS, 'p_static': 0.0}

for label, E in E_VALUES.items():
    res = validate_student_on_E(model, E, N_EPISODES, device, env_params_dyn)
    print_result(label + " [DYN]", res)
    results_dyn[label] = res

# ──────────────────────────────────────────────
# STATIC VALIDATION (p_static=1.0, 20 episodes)
# ──────────────────────────────────────────────
print("\n" + "="*70)
print("  STATIC VALIDATION (p_static=1.0)")
print("="*70)

N_STATIC_EP = 20
results_stat = {}
env_params_stat = {**ENV_PARAMS, 'p_static': 1.0}

for label, E in E_VALUES.items():
    res = validate_student_on_E(model, E, N_STATIC_EP, device, env_params_stat)
    print_result(label + " [STAT]", res)
    results_stat[label] = res


Device: cuda

Student caricato: ep 30, val_loss=0.24127
  obs_dim=44, action_dim=12

  MATCHED VALIDATION (p_static=0.30)


E=5.0e+06: 100%|██████████| 50/50 [07:35<00:00,  9.11s/it]



  soft [MATCHED]  (E = 5.0e+06)
  Episodi:             50
  Mean Error:          1.67 ± 2.45 cm
  Median Error:        1.01 cm
  P90 Error:           3.29 cm
  P95 Error:           4.21 cm
  Success @5mm:         12.3%
  Success @1cm:         49.5%
  Success @15mm:        72.0%
  Success @2cm:         80.5%
  OnGoal fraction:     0.495 ± 0.247


E=6.5e+06: 100%|██████████| 50/50 [07:46<00:00,  9.32s/it]



  soft_to_med [MATCHED]  (E = 6.5e+06)
  Episodi:             50
  Mean Error:          1.45 ± 2.47 cm
  Median Error:        0.84 cm
  P90 Error:           2.52 cm
  P95 Error:           3.75 cm
  Success @5mm:         18.9%
  Success @1cm:         61.5%
  Success @15mm:        79.2%
  Success @2cm:         86.2%
  OnGoal fraction:     0.615 ± 0.282


...


  medium [STAT]  (E = 1.0e+07)
  Episodi:             20
  Mean Error:          2.46 ± 2.54 cm
  Median Error:        1.71 cm
  P90 Error:           5.49 cm
  P95 Error:           5.74 cm
  Success @5mm:          3.2%
  Success @1cm:         23.5%
  Success @15mm:        41.5%
  Success @2cm:         55.6%
  OnGoal fraction:     0.235 ± 0.341


E=1.5e+07: 100%|██████████| 20/20 [03:07<00:00,  9.38s/it]



  med_to_rigid [STAT]  (E = 1.5e+07)
  Episodi:             20
  Mean Error:          2.61 ± 2.55 cm
  Median Error:        2.05 cm
  P90 Error:           4.51 cm
  P95 Error:           5.29 cm
  Success @5mm:          6.4%
  Success @1cm:         19.9%
  Success @15mm:        34.5%
  Success @2cm:         47.9%
  OnGoal fraction:     0.199 ± 0.317


E=2.0e+07: 100%|██████████| 20/20 [03:15<00:00,  9.77s/it]



  rigid [STAT]  (E = 2.0e+07)
  Episodi:             20
  Mean Error:          3.17 ± 3.43 cm
  Median Error:        2.09 cm
  P90 Error:           8.86 cm
  P95 Error:           11.33 cm
  Success @5mm:          8.1%
  Success @1cm:         17.2%
  Success @15mm:        28.7%
  Success @2cm:         42.3%
  OnGoal fraction:     0.172 ± 0.335

SUMMARY STUDENT — MATCHED vs DYNAMIC vs STATIC                                                                
Label              E            @1cm MATCH   @1cm DYN     @1cm STAT    OnGoal MATCH   OnGoal DYN   OnGoal STAT 
--------------------------------------------------------------------------------------------------------------
soft               5.0e+06          49.5%       61.1%       10.6%        0.495        0.611        0.106
soft_to_med (interp) 6.5e+06          61.5%       74.5%       19.9%        0.615        0.745        0.199
medium_soft        7.5e+06          59.5%       69.2%       20.8%        0.595        0.692        0.208
me

In [ ]:
# SUMMARY TABLE

print(f"\n{'='*110}")
print(f"{'SUMMARY STUDENT — MATCHED vs DYNAMIC vs STATIC':<110}")
print(f"{'='*110}")
print(f"{'Label':<18} {'E':<12} {'@1cm MATCH':<12} {'@1cm DYN':<12} {'@1cm STAT':<12} {'OnGoal MATCH':<14} {'OnGoal DYN':<12} {'OnGoal STAT':<12}")
print(f"{'-'*110}")
for label in E_VALUES:
    rm = results_match[label]
    rd = results_dyn[label]
    rs = results_stat[label]
    tag = " (interp)" if label in ("soft_to_med", "med_to_med", "med_to_rigid") else ""
    print(f"{label+tag:<18} {rd['E']:<12.1e} "
          f"{rm['success_1cm']*100:>8.1f}%   "
          f"{rd['success_1cm']*100:>8.1f}%   "
          f"{rs['success_1cm']*100:>8.1f}%   "
          f"{rm['on_goal_frac_mean']:>10.3f}     "
          f"{rd['on_goal_frac_mean']:>8.3f}     "
          f"{rs['on_goal_frac_mean']:>8.3f}")

In [ ]:
# SAVE JSON

results_json = {"matched": {}, "dynamic": {}, "static": {}}
for label in E_VALUES:
    results_json["matched"][label] = {
        k: v for k, v in results_match[label].items() if k != "trajectories"
    }
    results_json["dynamic"][label] = {
        k: v for k, v in results_dyn[label].items() if k != "trajectories"
    }
    results_json["static"][label] = {
        k: v for k, v in results_stat[label].items() if k != "trajectories"
    }
with open(OUTPUT_DIR / "student_validation.json", "w") as f:
    json.dump(results_json, f, indent=2)

In [ ]:
# Plots

for suffix, results in [("matched", results_match),
                        ("dyn",     results_dyn),
                        ("stat",    results_stat)]:
    plot_summary(results, OUTPUT_DIR, suffix=suffix)
    plot_error_trajectories(results, OUTPUT_DIR, suffix=suffix)


### Continuous generalization test

Runs the student against 100 episodes at random E values drawn
uniformly in log-space across `[E_MIN, E_MAX]`. Complements the
discrete validation above by probing the whole stiffness range
densely.

The episode-level success metric is **the fraction of steps with
error below 1 cm** (`success_frac_1cm`), NOT a last-step flag. This
matches the success metric used in the discrete validation, so the
two analyses are directly comparable.


In [16]:
def test_continuous_generalization(
        model_path: str = STUDENT_PATH,
        output_dir: Path = None,
        n_episodes: int = 100,
        seed: int = 42,
        ep_success_threshold: float = 0.5,
):
    """Random-E generalization test.

    Parameters
    ----------
    model_path           : path to the student .pth
    output_dir           : where to save plots and CSV (defaults to
                           a subfolder of the validation OUTPUT_DIR)
    n_episodes           : number of random-E episodes
    seed                 : seed for the random E draw (deterministic)
    ep_success_threshold : an episode is 'on-target' if its
                           success_frac_1cm exceeds this. 0.5 means
                           >50% of steps under 1 cm.
    """
    if output_dir is None:
        output_dir = OUTPUT_DIR / "continuous_generalization"
    output_dir.mkdir(parents=True, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load the student.
    ckpt = torch.load(model_path, map_location=device, weights_only=False)
    student = StudentPolicy(ckpt["obs_dim"], ckpt["action_dim"]).to(device)
    student.load_state_dict(ckpt["model_state_dict"])
    student.eval()

    print("\n--- Continuous Generalization Test ---")
    print(f"  Model      : {model_path}")
    print(f"  E range    : {E_MIN:.1e} -- {E_MAX:.1e} Pa (log scale)")
    print(f"  Episodes   : {n_episodes}")

    # Random E values, log-uniformly distributed. Deterministic given
    # `seed` so reruns produce comparable results.
    rng = np.random.default_rng(seed)
    random_Es = 10 ** rng.uniform(np.log10(E_MIN), np.log10(E_MAX),
                                  size=n_episodes)

    results = []
    for ep in tqdm(range(n_episodes), desc="Random-E episodes"):
        random_E   = float(random_Es[ep])
        E_norm     = normalize_E(random_E)
        env_params = {**ENV_PARAMS, "young_modulus": random_E}
        env        = RodTrackingEnv(**env_params)

        # Different prime multiplier (1009) than the discrete loop
        # (7919) so the random seeds don't overlap with the matched
        # validation episodes.
        obs, _ = env.reset(seed=seed + ep * 1009)

        done = False
        ep_errors = []
        while not done:
            action = student_predict(student, obs, E_norm, device)
            obs, _, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            ep_errors.append(info["error"])

        ep_errors_arr = np.array(ep_errors)
        results.append({
            "E":                random_E,
            "E_norm":           E_norm,
            "mean_error_cm":    float(np.mean(ep_errors_arr)   * 100),
            "median_error_cm":  float(np.median(ep_errors_arr) * 100),
            "p90_error_cm":     float(np.percentile(ep_errors_arr, 90) * 100),
            # success_frac_* are FRACTIONS in [0, 1] (consistent with
            # the discrete validation above).
            "success_frac_1cm": float(np.mean(ep_errors_arr < 0.01)),
            "success_frac_2cm": float(np.mean(ep_errors_arr < 0.02)),
        })
        env.close()

    df = pd.DataFrame(results).sort_values("E").reset_index(drop=True)

    # Aggregate statistics.
    mean_success_1cm = df["success_frac_1cm"].mean() * 100
    mean_success_2cm = df["success_frac_2cm"].mean() * 100
    episodes_success = (df["success_frac_1cm"] > ep_success_threshold).mean() * 100

    print("\n--- Aggregate results ---")
    print(f"  Mean step success @1cm        : {mean_success_1cm:.1f}%")
    print(f"  Mean step success @2cm        : {mean_success_2cm:.1f}%")
    print(f"  Episodes with >{ep_success_threshold * 100:.0f}% on-target: {episodes_success:.1f}%")
    print(f"  Mean error                    : {df['mean_error_cm'].mean():.2f} cm")
    print(f"  Median of episode medians     : {df['median_error_cm'].median():.2f} cm")

    csv_path = output_dir / "generalization_results.csv"
    df.to_csv(csv_path, index=False)
    print(f"\n  CSV saved: {csv_path}")

    # ---------------- Plot 1: mean error vs E (colored by success) -------
    fig, ax = plt.subplots(figsize=(11, 6.5))
    scatter = ax.scatter(
        df["E"], df["mean_error_cm"],
        c=df["success_frac_1cm"] * 100,
        cmap="RdYlGn", vmin=30, vmax=90,
        alpha=0.85, s=70, edgecolors="black", linewidths=0.5,
    )
    ax.axhline(1.0, color="black", linestyle="--", linewidth=1, alpha=0.7,
               label="1 cm target threshold")
    ax.axhline(2.0, color="gray", linestyle=":", linewidth=1, alpha=0.5,
               label="2 cm threshold")
    ax.set_xscale("log")

    # Mark the 4 trained experts.
    for E_train, lbl in TRAINED_E:
        ax.axvline(E_train, color="blue", linestyle=":", alpha=0.4, linewidth=1)
        ax.text(E_train, ax.get_ylim()[1] * 0.95, lbl,
                rotation=90, fontsize=8, color="blue",
                verticalalignment="top", horizontalalignment="right")

    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label("Success @1cm [%]", rotation=270, labelpad=20)

    ax.set_title(
        f"Continuous generalization - "
        f"mean step success @1cm: {mean_success_1cm:.1f}%   |   "
        f"episodes >{ep_success_threshold * 100:.0f}% on-target: "
        f"{episodes_success:.1f}%"
    )
    ax.set_xlabel("Young's modulus E [Pa] (log scale)")
    ax.set_ylabel("Mean error [cm]")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(loc="upper right")

    plt.tight_layout()
    plot1 = output_dir / "continuous_generalization_error.png"
    plt.savefig(plot1, dpi=150)
    plt.show()
    print(f"  Plot saved: {plot1}")

    # ---------------- Plot 2: success @1cm vs E + rolling mean -----------
    fig, ax = plt.subplots(figsize=(11, 6.5))
    ax.scatter(
        df["E"], df["success_frac_1cm"] * 100,
        c=df["mean_error_cm"], cmap="RdYlGn_r", vmin=0.5, vmax=2.0,
        alpha=0.85, s=70, edgecolors="black", linewidths=0.5,
    )
    ax.set_xscale("log")
    ax.axhline(50, color="gray", linestyle="--", alpha=0.5,
               label="50% success threshold")

    for E_train, lbl in TRAINED_E:
        ax.axvline(E_train, color="blue", linestyle=":", alpha=0.4, linewidth=1)
        ax.text(E_train, 10, lbl, rotation=90, fontsize=8, color="blue",
                verticalalignment="bottom", horizontalalignment="right")

    # Rolling-mean trend line over E. Window size=10 is a balance:
    # too small -> noisy, too large -> washes out the dip near
    # interpolation points.
    window  = 10
    rolling = (df["success_frac_1cm"]
               .rolling(window=window, center=True).mean() * 100)
    ax.plot(df["E"], rolling, color="darkblue", linewidth=2,
            label=f"Rolling mean ({window} eps)", alpha=0.7)

    ax.set_title("Step-wise success rate @1cm across continuous E range")
    ax.set_xlabel("Young's modulus E [Pa] (log scale)")
    ax.set_ylabel("Success fraction @1cm [%]")
    ax.set_ylim(0, 100)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(loc="lower right")

    plt.tight_layout()
    plot2 = output_dir / "continuous_generalization_success.png"
    plt.savefig(plot2, dpi=150)
    plt.show()
    print(f"  Plot saved: {plot2}")

    return df


NameError: name 'MODEL_PATH' is not defined

In [ ]:
df_gen = test_continuous_generalization()
df_gen.head()
